# 01 - Bronze ingestion

Mengambil data USGS dan Open-Meteo, lalu menulis bronze JSON ke ADLS.

In [ ]:
window_start = "2015-10-01"
window_end = "2015-10-31"

In [ ]:
# Azure and source configuration
import json
import os
import time
from datetime import date, timedelta

from notebookutils import mssparkutils
import requests

STORAGE_ACCOUNT = "sthidrogkiq7v7oc3lns"
BRONZE_ROOT = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net"
USGS_BASE = "https://api.waterdata.usgs.gov/ogcapi/v1/collections"
METEO_URL = "https://archive-api.open-meteo.com/v1/archive"

USGS_API_KEY = os.getenv("USGS_API_KEY", "").strip()
try:
    USGS_API_KEY = mssparkutils.credentials.getSecret(
        "kv-hidro-gkiq7v7oc3lns", "usgs-api-key"
    )
except Exception:
    pass

STATIONS = [
    {"site": "11525670", "lat": 40.65178087, "lon": -122.9144343},
    {"site": "11150500", "lat": 35.9302434, "lon": -120.8687915},
    {"site": "11454000", "lat": 38.5151562, "lon": -122.0818793},
]

METEO_VARIABLES = [
    "precipitation",
    "rain",
    "soil_moisture_0_to_7cm",
    "soil_moisture_7_to_28cm",
    "soil_moisture_28_to_100cm",
    "temperature_2m",
    "relative_humidity_2m",
    "surface_pressure",
    "et0_fao_evapotranspiration",
    "wind_speed_10m",
    "shortwave_radiation",
    "cloud_cover",
]


def date_range(start, end):
    current = date.fromisoformat(start)
    stop = date.fromisoformat(end)
    while current <= stop:
        yield current.isoformat()
        current += timedelta(days=1)


def get_json(url, params, headers=None, retries=3):
    for attempt in range(retries):
        response = requests.get(url, params=params, headers=headers, timeout=90)
        if response.status_code == 429:
            time.sleep(5 * (attempt + 1))
            continue
        response.raise_for_status()
        return response.json()
    response.raise_for_status()


def fetch_usgs(collection, site, parameter_code, start, end):
    url = f"{USGS_BASE}/{collection}/items"
    params = {
        "monitoring_location_id": f"USGS-{site}",
        "parameter_code": parameter_code,
        "datetime": f"{start}T00:00:00Z/{end}T23:59:59Z",
        "limit": 50000,
        "f": "json",
    }
    headers = {"X-Api-Key": USGS_API_KEY} if USGS_API_KEY else {}
    features = []
    while url:
        payload = get_json(url, params, headers)
        features.extend(payload.get("features", []))
        next_link = next(
            (link["href"] for link in payload.get("links", []) if link.get("rel") == "next"),
            None,
        )
        if next_link:
            url = next_link
            params = None
        else:
            url = None
    return features


def write_usgs(prefix, features, start, end):
    written = 0
    for day in date_range(start, end):
        daily = [
            feature
            for feature in features
            if str(feature.get("properties", {}).get("time", ""))[:10] == day
        ]
        path = f"{BRONZE_ROOT}/{prefix}/ingest_date={day}/part-000.json"
        if mssparkutils.fs.exists(path):
            continue
        payload = {"features": daily}
        mssparkutils.fs.put(path, json.dumps(payload), True)
        written += 1
    return written


def fetch_meteo(latitude, longitude, start, end):
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start,
        "end_date": end,
        "hourly": ",".join(METEO_VARIABLES),
        "timezone": "UTC",
    }
    return get_json(METEO_URL, params)


def write_meteo(site, payload, start, end):
    times = payload["hourly"]["time"]
    written = 0
    for day in date_range(start, end):
        indexes = [index for index, value in enumerate(times) if value.startswith(day)]
        if not indexes:
            continue
        daily = {
            "latitude": payload.get("latitude"),
            "longitude": payload.get("longitude"),
            "elevation": payload.get("elevation"),
            "timezone": payload.get("timezone", "UTC"),
            "hourly_units": payload.get("hourly_units", {}),
            "hourly": {
                "time": [times[index] for index in indexes],
                **{
                    variable: [payload["hourly"][variable][index] for index in indexes]
                    for variable in METEO_VARIABLES
                    if variable in payload["hourly"]
                },
            },
        }
        path = f"{BRONZE_ROOT}/open-meteo/station_id={site}/ingest_date={day}/part-000.json"
        if mssparkutils.fs.exists(path):
            continue
        mssparkutils.fs.put(path, json.dumps(daily), True)
        written += 1
    return written


total = 0
for station in STATIONS:
    site = station["site"]
    for collection, parameter_code in (
        ("continuous", "00065"),
        ("continuous", "00060"),
        ("daily", "00060"),
    ):
        features = fetch_usgs(collection, site, parameter_code, window_start, window_end)
        prefix = f"usgs/{collection}/parameter_code={parameter_code}/station_id={site}"
        count = write_usgs(prefix, features, window_start, window_end)
        total += count
        print(f"{site} {collection} {parameter_code}: {len(features)} features, {count} files")

    meteo = fetch_meteo(station["lat"], station["lon"], window_start, window_end)
    count = write_meteo(site, meteo, window_start, window_end)
    total += count
    print(f"{site} open-meteo: {count} files")

print(f"bronze files written: {total}")
